# L4a: Trading Rules Using Binomial Lattice Models
In this lecture, we continue our exploration of binomial lattice models of share price dynamics. We will use net present value (NPV) to evaluate a long stock position and compute the probability of exceeding a target return at a scheduled sale time. This is our first trading-focused lecture!

> __Learning Objectives__
>
> By the end of this lecture, you will be able to:
> 
> * **Evaluate a stock trade using net present value.** Derive the scaled NPV of a long position and explain how the sale price, holding period, and benchmark growth rate affect the result.
> * **Compute terminal target probabilities using a binomial lattice.** Find the minimum number of up moves needed to exceed a target scaled NPV, then calculate the probability of that outcome.
> * **Extend the binomial lattice to allow $m$ possible price outcomes at each step.** Construct an N-ary lattice and use branch counts to calculate its node prices, probabilities, and number of states at each level.

This is going to be a fun lecture! Let's get started!

___

## Examples
Today, we will use two examples to connect the lecture's probability formulas to data, code and decision-making:

> [▶ Explore a terminal target probability](CHEME-5660-L4a-Example-CumulativeProbabilityLattice-Fall-2026.ipynb). In this example, we estimate lattice parameters from historical data and compute the probability that the scaled NPV strictly exceeds $\rho_\star$ at the scheduled sale time. We also compute the complementary probability of finishing at or below the target.

> [▶ Explore N-ary lattice models](CHEME-5660-L4a-Example-N-Ary-Lattice-Fall-2026.ipynb). In this example, we estimate several branch factors and probabilities from historical growth rates, build a recombining lattice, and compute its node prices and multinomial probabilities. We check that the probabilities sum to one and plot the resulting distributions.

Optional extensions are collected near the end of the lecture.

___

## From the Bid–Ask Spread to a Lattice Price
The [L3a lecture](../../week-3/L3a/CHEME-5660-L3a-Lecture-Equity-Exchanges-Return-StylizedFacts-Fall-2026.ipynb) introduced exchanges, order types, order books, and the National Best Bid and Offer (NBBO). We need only one piece of that machinery today: the price used in a forecast is not automatically the price at which a trade executes.

> __Company profile: Citadel Securities.__ [Citadel Securities](https://www.citadelsecurities.com/who-we-are/) is a global market maker that combines quantitative research with trading technology. Its [equities business](https://www.citadelsecurities.com/what-we-do/equities/) executes trades for brokers and institutional investors. What does a market maker do? It provides liquidity by standing ready to buy from sellers and sell to buyers. In doing so, it can earn the [bid–ask spread](https://www.citadelsecurities.com/what-we-do/what-is-a-market-maker/), while taking the risk that prices move against its positions.
>
> __Explore further:__ Visit their [YouTube channel](https://www.youtube.com/@citadelsecurities), learn about [internships](https://www.citadelsecurities.com/careers/internships/), or browse [open positions](https://www.citadelsecurities.com/careers/open-opportunities/).

Let's connect this business to the prices in our model. At time $t$, a market maker may quote a bid $b_t$, the price at which it will buy, and an ask $a_t$, the price at which it will sell. The bid–ask spread $s_{t}$ and midpoint $m_{t}$ are given by:
$$
s_t=a_t-b_t,\qquad m_t=\frac{a_t+b_t}{2}.
$$

In the NPV model we introduce in lecture, we will replace the executable bid and ask quotes $(b_t,a_t)$ with a single frictionless price $S_t$. The [execution-aware extension](advanced/execution/CHEME-5660-L4a-Advanced-ExecutionAware-ProbabilityOfProfit-Fall-2026.ipynb) restores the spread, fees, and slippage.

With this simplification, let's review how the binomial lattice assigns probabilities to possible future prices.

### Review: The Binomial Lattice Model
A binomial lattice represents an asset price on a discrete time grid. From each state, the next price moves by one of two multiplicative factors. Here, $t=0,1,\ldots$ counts lattice steps, and $S_t$ denotes the price after $t$ steps. Each step lasts $\Delta t>0$ years, so the elapsed time at level $t$ is $t\Delta t$ years.

<div>
    <center>
        <img src="figs/Fig-Lattice-Schematic.svg" width="800" alt="Three panels showing one-, two-, and three-step binomial lattices in which up and down branches recombine"/>
    </center>
</div>

Let $S_0>0$ be the initial price (root of the tree). At each step, we flip a coin, i.e., we perform a Bernoulli trial, and multiply the price by $u$ with real-world probability $p$ or by $d$ with probability $1-p$, where $u>d>0$ and $0<p<1$. The up branch increases the price when $u>1$, while the down branch decreases the price when $d<1$.

We assume successive coin flips are independent and hold $(u,d,p)$ fixed across time, so the branch choices are independent and identically distributed.

> __What distribution does a constant-parameter binomial lattice imply?__
>
> Let $K_t$ be the number of up moves in the first $t$ steps. Then $K_t\sim\operatorname{Binomial}(t,p)$. Each up-move count gives a different price, so the probability of a level-$t$ price node is given by:
>
> $$
> \boxed{
> \mathbb P\!\left(S_t=S_0u^kd^{t-k}\right)
> =\binom{t}{k}p^k(1-p)^{t-k},
> \qquad k=0,1,\ldots,t.
> }
> $$
>
> The $t+1$ possible up-move counts therefore give a complete finite distribution for the share price after $t$ steps.

We use this distribution to forecast possible prices under the real-world probability measure $\mathbb P$. The parameters $(u,d,p)$ may differ across assets or calibration windows.

With independent moves and fixed parameters, this model does not capture volatility clustering. Later in the course, we will also use lattices to price derivatives under the risk-neutral probability measure $\mathbb{Q}$ introduced in the [L3b lecture](../../week-3/L3b/CHEME-5660-L3b-Lecture-LatticeModels-Equity-Price-Fall-2026.ipynb).

___


## Theory: Net Present Value (NPV) Trade Rule

Let's use our abstract asset framework to evaluate a stock trade. We will start
with the cash flows from buying and selling shares, then use the net present value
(NPV) to account for the time between these two events.

> __Scenario:__ Suppose we purchase $n_0>0$ shares of ticker `XYZ` at time $0$ for
> $S_0>0$ USD/share. We hold the shares for $N$ time steps and sell all of them at
> time $T=N\Delta t$ for $S_T$ USD/share. Here, $N$ is a positive integer and
> $\Delta t>0$ is the duration of each step in years, so $T$ is the holding period
> in years. We assume no dividends, transaction fees, or bid–ask spread.

This is a __long position__: we buy shares with the expectation that their price
will increase. From our perspective, the purchase is a cash outflow of $n_0S_0$
USD, and the sale is a cash inflow of $n_0S_T$ USD. These are the two cash-flow
events that make up our abstract asset.

Following the
[L1b lecture](../../week-1/L1b/CHEME-5660-L1b-Lecture-TimeValueMoney-Fall-2026.ipynb),
let $g_y$ denote the constant, continuously compounded annual benchmark growth
rate associated with the selected yield $y$. We may use a risk-free rate or an
assumed growth rate for another investment as our benchmark. The units of $g_y$
are inverse years, so $e^{-g_yT}$ is a dimensionless discount factor. Multiplying
the sale proceeds by this factor gives their present value. Thus, the NPV of the
trade is given by:

$$
\operatorname{NPV}(g_y,T)
=\underbrace{-n_0S_0}_{\text{purchase today}}
+\underbrace{n_0S_Te^{-g_yT}}_{\text{present value of sale proceeds}}.
$$

The NPV is measured in today's USD and scales with the number of shares we buy.
To express the result relative to our initial investment, let's divide by
$n_0S_0$ and denote the resulting __scaled NPV__ by $\rho_T$:

$$
\begin{aligned}
\rho_T
&=\frac{\operatorname{NPV}(g_y,T)}{n_0S_0}\\
&=\frac{-n_0S_0+n_0S_Te^{-g_yT}}{n_0S_0}\\
&=\left(\frac{S_T}{S_0}\right)e^{-g_yT}-1.
\end{aligned}
$$

> __What does the scaled NPV tell us?__ The quantity $\rho_T$ is a dimensionless
> discounted fractional return on our initial investment. If $\rho_T>0$, the
> present value of the sale proceeds exceeds the purchase cost. If $\rho_T=0$,
> the two are equal; if $\rho_T<0$, the discounted proceeds fall short. Dividing
> by the initial investment removes the dependence on the number of shares,
> allowing us to compare trades of different sizes.

The trade schematic shows how the NPV per share, $S_0\rho_T$, changes with the
sale price. The vertical axis is measured in USD/share; dividing its values by
the purchase price $S_0$ gives the dimensionless scaled NPV $\rho_T$.

<div>
    <center>
        <img src="figs/Fig-TradeRule-Schematic.svg" width="800" alt="Discounted per-share NPV versus terminal share price, showing the break-even threshold"/>
    </center>
</div>

Let's examine the role of the holding period before connecting this expression
to our lattice model.

### Short holding periods

For a holding period of a few trading days, the discount factor may be close to
one. The condition we need is $|g_y|T\ll1$. Under this approximation, the scaled
NPV becomes:

$$
\rho_T\approx\frac{S_T}{S_0}-1
=\frac{S_T-S_0}{S_0}.
$$

This is the familiar fractional change in the share price. For example, buying
at 100 USD/share and selling at 105 USD/share gives a fractional price return of
0.05, or 5%. When discounting has little effect over the holding period, the
scaled NPV is approximately this same value.

### Longer holding periods

As the holding period grows, discounting can have a substantial effect. We can
see this by asking which sale price makes the NPV zero. Setting $\rho_T=0$ and
solving for $S_T$ gives:

$$
\left(\frac{S_T}{S_0}\right)e^{-g_yT}-1=0
\quad\Longrightarrow\quad S_T=S_0e^{g_yT}.
$$

The break-even sale price is the initial share price grown at the benchmark rate
over the holding period. For $g_y>0$, a price increase alone is not enough: the
sale price must exceed $S_0e^{g_yT}$ for the NPV to be positive. We will retain the exact discounted expression
when computing trade probabilities, so the same calculation applies to both
short and long holding periods.

> __Why is this useful?__ At the time we buy the shares, the future sale price
> $S_T$ is unknown. Our binomial lattice assigns probabilities to its possible
> values. Each possible sale price gives a corresponding scaled NPV through the
> expression we just derived, with the same probability.

We can therefore use the lattice to ask: what is the probability that the scaled
NPV exceeds a specified target at the end of the holding period? Let's develop
that calculation next.

___


## Terminal Target Probability

Our NPV expression tells us the discounted fractional return for a given sale
price. Let's now use the binomial lattice to compute the probability that this
return exceeds a target. We will evaluate the trade at the scheduled sale time
$T=N\Delta t$; this is what we mean by a __terminal__ target.

> __Scenario:__ Suppose we choose a holding period $T=N\Delta t$ and a target
> scaled NPV $\rho_\star$. For example, $\rho_\star=0.05$ means that we want the
> present value of the sale proceeds to exceed the initial investment by more
> than 5%. Our success event is $\rho_T>\rho_\star$: a return exactly equal to
> the target does not count as exceeding it.

Several terminal prices may satisfy this condition. To compute the probability
of success, we need to identify those prices and add their probabilities. The
binomial lattice makes this calculation possible because each terminal price
is determined by the number of up moves.

### From the number of up moves to the scaled NPV

Let $K_N$ denote the number of up moves during the $N$ lattice steps. As in our
lattice review, suppose the moves are independent, with fixed factors $u>d>0$
and up-move probability $p\in(0,1)$. Then $K_N\sim\operatorname{Binomial}(N,p)$.
If we observe $k$ up moves, there must be $N-k$ down moves, giving the terminal
price:

$$
S_T=S_0u^kd^{N-k},\qquad k=0,1,\ldots,N.
$$

Substituting this price into the scaled NPV expression gives the return
associated with $k$ up moves:

$$
\begin{aligned}
\rho(k)
&=\left(\frac{S_0u^kd^{N-k}}{S_0}\right)e^{-g_yN\Delta t}-1\\
&=u^kd^{N-k}e^{-g_yN\Delta t}-1.
\end{aligned}
$$

Thus, the random scaled NPV is $\rho_T=\rho(K_N)$. The holding period and
benchmark growth rate $g_y$ are fixed for this calculation; the uncertainty
comes from the number of up moves.

> __Why count up moves?__ At a fixed level of the tree, increasing $k$ by one
> replaces a down factor $d$ with an up factor $u$. The terminal price is
> multiplied by $u/d>1$, so the price and its scaled NPV both increase. Once a
> particular up-move count exceeds the target, every larger count does too.

We therefore need to find the smallest number of up moves that exceeds the
target. Let's solve for that number.

### Finding the up-move threshold

First, consider a target $\rho_\star>-1$, so $1+\rho_\star$ is positive. Starting
with the desired return inequality, add one to both sides and remove the
discount factor:

$$
\begin{aligned}
u^kd^{N-k}e^{-g_yN\Delta t}-1&>\rho_\star\\
u^kd^{N-k}e^{-g_yN\Delta t}&>1+\rho_\star\\
u^kd^{N-k}&>(1+\rho_\star)e^{g_yN\Delta t}.
\end{aligned}
$$

Both sides are positive. Taking the natural logarithm preserves the inequality
and lets us bring the powers down as coefficients:

$$
k\ln u+(N-k)\ln d>\ln(1+\rho_\star)+g_yN\Delta t.
$$

Expanding the $(N-k)\ln d$ term and collecting the terms containing $k$ gives:

$$
\begin{aligned}
k(\ln u-\ln d)+N\ln d
&>\ln(1+\rho_\star)+g_yN\Delta t\\
k\ln(u/d)
&>\ln(1+\rho_\star)+g_yN\Delta t-N\ln d.
\end{aligned}
$$

Since $\ln(u/d)>0$, dividing by it also preserves the inequality. We obtain
$k>\tau(\rho_\star)$, where the real-valued up-move threshold is:

$$
\boxed{
\tau(\rho_\star)=
\frac{\ln(1+\rho_\star)+g_yN\Delta t-N\ln d}{\ln(u/d)}.
}
$$

The number of up moves must be an integer. Therefore, the smallest integer
strictly greater than this threshold is:

$$
\kappa=\lfloor\tau(\rho_\star)\rfloor+1,
$$

where $\lfloor x\rfloor$ is the greatest integer less than or equal to $x$.
For example, if $\tau=2.4$, we need at least three up moves. If $\tau=2$
exactly, we still need three: two up moves would reach the target exactly,
while our rule requires exceeding it.

### Computing the cumulative probability

There are only $N+1$ possible up-move counts, from $0$ through $N$. If
$1\leq\kappa\leq N$, the successful counts are $\kappa,\kappa+1,\ldots,N$.
Two other cases have a direct interpretation: if $\kappa\leq0$, every terminal
node exceeds the target; if $\kappa>N$, none does.

We record these cases using the cutoff $k_{\min}$:

$$
k_{\min}=\min\!\left\{\max\!\left\{\kappa,0\right\},N+1\right\}.
$$

This leaves attainable cutoffs unchanged, uses $0$ when all nodes succeed, and
uses $N+1$ to indicate that no node succeeds. The value $N+1$ is a bookkeeping
convention; the tree still has at most $N$ up moves.

> __Terminal target probability:__ Under the fixed-parameter, independent-move
> binomial model above, for $\rho_\star>-1$, the probability that the scaled NPV
> exceeds the target at time $T=N\Delta t$ is:
> $$
> \begin{aligned}
> \mathbb P(\rho_T>\rho_\star)
> &=\mathbb P(K_N\geq k_{\min})\\
> &=\begin{cases}
> 1, & k_{\min}=0,\\
> \displaystyle\sum_{k=k_{\min}}^N\binom Nk p^k(1-p)^{N-k},
> & 1\leq k_{\min}\leq N,\\
> 0, & k_{\min}=N+1.
> \end{cases}
> \end{aligned}
> $$
> Each term in the sum is the probability of one successful up-move count.
> Adding these terms gives the __binomial upper tail__, the probability of
> observing at least $k_{\min}$ up moves.

__Example: A three-month trade__

Suppose we hold the stock for three monthly steps, so $N=3$,
$\Delta t=1/12$ year, and $T=1/4$ year. For illustration, use $u=1.10$,
$d=0.90$, $p=0.6$, and a benchmark growth rate $g_y=0.04$ year$^{-1}$.
What is the probability that the scaled NPV exceeds 5%, i.e.,
$\rho_\star=0.05$?

Substituting these values into the up-move threshold expression gives:

$$
\tau(0.05)
=\frac{\ln(1.05)+0.04(3/12)-3\ln(0.90)}{\ln(1.10/0.90)}
\approx1.868.
$$

The number of up moves must be an integer strictly greater than this threshold,
so $\kappa=k_{\min}=2$. We can check this against the returns: one up move gives
$\rho(1)\approx-0.1179$, while two give $\rho(2)\approx0.0782$. Thus, two up
moves exceed our 5% target, while one does not. The successful counts are
$K_3=2$ and $K_3=3$, and their combined probability is given by:

$$
\begin{aligned}
\mathbb P(\rho_T>0.05)
&=\mathbb P(K_3\geq2)\\
&=\binom32(0.6)^2(0.4)+\binom33(0.6)^3\\
&=0.432+0.216=0.648.
\end{aligned}
$$

The model therefore assigns a 64.8% probability to exceeding the 5% target at
the scheduled sale time, three months after purchase. The return inequality
determines which nodes count as successes; $p$ determines how much probability
those nodes carry.

For targets $\rho_\star\leq-1$, we do not need the logarithmic calculation.
Every terminal price in this model is positive, so every scaled NPV is strictly
greater than $-1$. The success probability is therefore one.

### What about falling short of the target?

At the scheduled sale time, the scaled NPV either exceeds the target or is at
or below it. These events are complementary, giving:

$$
\mathbb P(\rho_T\leq\rho_\star)
=1-\mathbb P(\rho_T>\rho_\star).
$$

For example, setting $\rho_\star=-0.05$ in this expression gives the probability
that the discounted fractional return is at or below $-5\%$ at time $T$. This
checks the outcome on the scheduled sale date. Selling as soon as a loss boundary
is reached requires tracking the price path; we explore that calculation in the
[optional first-passage example](advanced/first-passage/CHEME-5660-L4a-Advanced-FirstPassage-ExitRules-Fall-2026.ipynb).

The benchmark and holding period also matter. Increasing $g_y$ while holding
the lattice and target fixed raises the return hurdle, so the success
probability cannot increase. Changing the number of steps $N$ changes both the
threshold and the distribution of $K_N$; we must recompute the probability to
determine the effect of a longer holding period.

Let's apply this calculation using parameters estimated from historical data.

> __Example:__
>
> [▶ Explore a terminal target probability](CHEME-5660-L4a-Example-CumulativeProbabilityLattice-Fall-2026.ipynb).
> Estimate the lattice parameters from historical data, identify the terminal
> nodes whose scaled NPV strictly exceeds $\rho_\star$, and add their
> probabilities. We also compute the complementary probability of finishing
> at or below the target.

Let's now extend the model to include more than two possible price movements at
each step.

___


## N-Ary Lattice Models

One limitation of the binomial lattice model is that it allows only two possible
price movements at each step. Suppose we want to include an unchanged price, or
distinguish small price movements from large ones.

> __Idea:__ We can extend the binomial model to an __N-ary lattice__ by specifying
> several possible price-change factors and a probability for each one. We will
> use $m$ for the number of branches, keeping $N$ for the number of steps in the
> holding period.

For our trading calculation, we still need the terminal prices and their
probabilities. Each price gives a scaled NPV, so we can identify the outcomes
that exceed our target and add their probabilities. Let's start with a
three-outcome example before writing the general model.

### A three-outcome price model

Suppose the current price is $S_0=100$ USD/share. At each step, the price can
increase by 10%, remain unchanged, or decrease by 10%. For illustration, assign
these outcomes the following factors and probabilities:

<div style="margin: 0.8em 0; overflow-x: auto;">
  <table aria-label="Three-outcome one-step price model" style="font-size: inherit; width: 100%; margin: 0; border-collapse: collapse; font-variant-numeric: tabular-nums; color: #24292f;">
    <thead>
      <tr>
        <th scope="col" style="background: #b31b1b; color: #ffffff; padding: 8px 12px; text-align: center; vertical-align: middle; font-size: 0.92em; font-weight: 600; line-height: 1.35; border: 0;">Branch <i>j</i></th>
        <th scope="col" style="background: #b31b1b; color: #ffffff; padding: 8px 12px; text-align: left; vertical-align: middle; font-size: 0.92em; font-weight: 600; line-height: 1.35; border: 0;">Price movement</th>
        <th scope="col" style="background: #b31b1b; color: #ffffff; padding: 8px 12px; text-align: right; vertical-align: middle; font-size: 0.92em; font-weight: 600; line-height: 1.35; border: 0;">Factor <i>f</i><sub><i>j</i></sub></th>
        <th scope="col" style="background: #b31b1b; color: #ffffff; padding: 8px 12px; text-align: right; vertical-align: middle; font-size: 0.92em; font-weight: 600; line-height: 1.35; border: 0;">Probability <i>p</i><sub><i>j</i></sub></th>
        <th scope="col" style="background: #b31b1b; color: #ffffff; padding: 8px 12px; text-align: right; vertical-align: middle; font-size: 0.92em; font-weight: 600; line-height: 1.35; border: 0;">Price after one step (USD/share)</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td style="background: #ffffff; padding: 7px 12px; text-align: center; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">1</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: left; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">Up by 10%</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">1.10</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">0.25</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">110</td>
      </tr>
      <tr>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: center; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">2</td>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: left; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">Unchanged</td>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">1.00</td>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">0.50</td>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">100</td>
      </tr>
      <tr>
        <td style="background: #ffffff; padding: 7px 12px; text-align: center; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 0;">3</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: left; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 0;">Down by 10%</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 0;">0.90</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 0;">0.25</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 0;">90</td>
      </tr>
    </tbody>
  </table>
</div>

After two steps of a three-branch tree, there are $3^2=9$ possible ordered paths. However, some paths
give the same final price. For example, an up move followed by a down move gives:

$$
100\times1.10\times0.90=99\text{ USD/share}.
$$

Reversing the order gives $100\times0.90\times1.10=99$ USD/share as well. Both paths have the same branch counts, so we can represent them by one node. This merging of paths is what we mean by
__recombination__.

> __What do we need to remember?__ To compute the price at a given level, we
> need only the number of times each branch has occurred. For example, in our three-outcome model,
> we write $\mathbf{x}=(x_1,x_2,x_3)$, where the entries count up, unchanged, and
> down moves, respectively. The vector records the counts, but we drop the order of the moves.

The intermediate prices along these paths differ, but their final prices are
the same. For the terminal-price calculations in this lecture, those counts
give us the information we need.

### Computing a node's price

Now consider $m\geq2$ possible branches. Branch $j$ has a positive,
dimensionless price factor $f_j$ and probability $p_j>0$, with
$\sum_{j=1}^{m}p_j=1$. At level $t$, let $x_j$ be the number of times branch
$j$ has occurred. Here, $t$ is a nonnegative integer counting lattice steps.
The elapsed time is $t\Delta t$ years.

Each step contributes to exactly one branch count. Therefore, the entries of
$\mathbf{x}=(x_1,\ldots,x_m)$ are nonnegative integers satisfying:

$$
x_1+x_2+\cdots+x_m=t.
$$

Every occurrence of branch $j$ multiplies the price by $f_j$. If it occurs
$x_j$ times, its contribution is $f_j^{x_j}$. Multiplying the contributions
from all branches gives the price at this node:

$$
\boxed{
S_t(\mathbf{x})=S_0f_1^{x_1}f_2^{x_2}\cdots f_m^{x_m}
=S_0\prod_{j=1}^{m}f_j^{x_j}.
}
$$

For the two-step state $\mathbf{x}=(1,0,1)$, this expression gives
$S_2(1,0,1)=100(1.10)^1(1.00)^0(0.90)^1=99$ USD/share, as expected.

__Aside__: This also recovers our binomial price formula. Let the number of branches be $m=2$, set $f_1=u$, $f_2=d$, and $\mathbf{x}=(k,t-k)$. Then the price is $S_t=S_0u^kd^{t-k}$.

### Computing a node's probability

We assume independent branch choices, using the same factors and probabilities
at every step. The probability of one particular ordered path is then the product
of its branch probabilities. 

In our ternary ($m=3$) example, the probability of an up move followed by a
down move is $0.25\times0.25=0.0625$. The reversed path has the same
probability. Because both paths reach the state $(1,0,1)$, we add their
probabilities:

$$
\mathbb P\!\left(\mathbf{X}_2=(1,0,1)\right)
=0.0625+0.0625=0.125,
$$

where $\mathbf{X}_t$ denotes the random vector of branch counts after $t$ steps.

Let's generalize this idea. Every ordered path with counts
$\mathbf{x}$ has probability $\prod_{j=1}^{m}p_j^{x_j}$. However, there are multiple
ways to achieve the same set of counts. The number of different orderings with those counts is given by the __multinomial coefficient__:

$$
\frac{t!}{x_1!x_2!\cdots x_m!}.
$$

Multiplying the number of distinct paths by the probability of each path gives:

> __Multinomial node probability:__ For independent branch choices with fixed
> probabilities, the probability of observing a count vector $\mathbf{x}$ at
> level $t$ is given by the multinomial distribution:
> $$
> \boxed{
> \mathbb P(\mathbf{X}_t=\mathbf{x})
> =\frac{t!}{x_1!\cdots x_m!}\prod_{j=1}^{m}p_j^{x_j},
> \qquad \sum_{j=1}^{m}x_j=t.
> }
> $$
> When there are two branches, the multinomial coefficient becomes
> $t!/[k!(t-k)!]=\binom{t}{k}$, and the expression reduces to the binomial
> probability $\binom{t}{k}p^k(1-p)^{t-k}$ that we saw with a binomial lattice.

We can now calculate the complete second level of our three-outcome example:

<div style="margin: 0.8em 0; overflow-x: auto;">
  <table aria-label="Two-step recombining lattice states" style="font-size: inherit; width: 100%; margin: 0; border-collapse: collapse; font-variant-numeric: tabular-nums; color: #24292f;">
    <thead>
      <tr>
        <th scope="col" style="background: #b31b1b; color: #ffffff; padding: 8px 12px; text-align: center; vertical-align: middle; font-size: 0.92em; font-weight: 600; line-height: 1.35; border: 0;">Count vector <b>x</b></th>
        <th scope="col" style="background: #b31b1b; color: #ffffff; padding: 8px 12px; text-align: left; vertical-align: middle; font-size: 0.92em; font-weight: 600; line-height: 1.35; border: 0;">Moves represented</th>
        <th scope="col" style="background: #b31b1b; color: #ffffff; padding: 8px 12px; text-align: right; vertical-align: middle; font-size: 0.92em; font-weight: 600; line-height: 1.35; border: 0;">Number of ordered paths</th>
        <th scope="col" style="background: #b31b1b; color: #ffffff; padding: 8px 12px; text-align: right; vertical-align: middle; font-size: 0.92em; font-weight: 600; line-height: 1.35; border: 0;">Price (USD/share)</th>
        <th scope="col" style="background: #b31b1b; color: #ffffff; padding: 8px 12px; text-align: right; vertical-align: middle; font-size: 0.92em; font-weight: 600; line-height: 1.35; border: 0;">Probability</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td style="background: #ffffff; padding: 7px 12px; text-align: center; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">(2,0,0)</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: left; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">Two up moves</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">1</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">121</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">0.0625</td>
      </tr>
      <tr>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: center; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">(1,1,0)</td>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: left; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">One up, one unchanged</td>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">2</td>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">110</td>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">0.2500</td>
      </tr>
      <tr>
        <td style="background: #ffffff; padding: 7px 12px; text-align: center; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">(1,0,1)</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: left; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">One up, one down</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">2</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">99</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">0.1250</td>
      </tr>
      <tr>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: center; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">(0,2,0)</td>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: left; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">Two unchanged moves</td>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">1</td>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">100</td>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">0.2500</td>
      </tr>
      <tr>
        <td style="background: #ffffff; padding: 7px 12px; text-align: center; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">(0,1,1)</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: left; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">One unchanged, one down</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">2</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">90</td>
        <td style="background: #ffffff; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 1px solid #e8eaed;">0.2500</td>
      </tr>
      <tr>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: center; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 0;">(0,0,2)</td>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: left; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 0;">Two down moves</td>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 0;">1</td>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 0;">81</td>
        <td style="background: #f7f7f7; padding: 7px 12px; text-align: right; vertical-align: middle; font-weight: 400; line-height: 1.25; border: 0; border-bottom: 0;">0.0625</td>
      </tr>
    </tbody>
  </table>
</div>

The nine ordered paths have recombined into six count states. Their
probabilities sum to one, so this table is a complete price distribution
after two steps.

### How many states do we need?

Our three-outcome example has nine ordered paths after two steps, but we only
need six nodes to represent their branch counts. How can we find this number
without listing all the paths?

Each node corresponds to a count vector $\mathbf{x}=(x_1,\ldots,x_m)$.
Thus, the number of nodes at level $t$ is the number of ways to choose
nonnegative integer counts satisfying $x_1+\cdots+x_m=t$.

Let's picture these counts using stars for the moves and bars to separate
the branches. In our three-outcome example, the state $(1,0,1)$ can be written as:

$$
\star\;|\;|\;\star \quad\longleftrightarrow\quad (1,0,1).
$$

There is one star in the first group, none between the two bars, and one in
the last group. These groups record one up move, no unchanged moves, and one
down move. Moving the bars changes the counts: placing both bars after the
two stars gives $(2,0,0)$, or two up moves.

For $t$ steps and $m$ branches, we arrange $t$ stars and $m-1$ bars.
There are $t+m-1$ positions in total. Once we choose which $m-1$ positions
hold the bars, the remaining positions hold the stars, giving one count
vector. This is the __stars-and-bars__ argument, and it gives the number
of states at level $t$:

$$
\boxed{L_t=\binom{t+m-1}{m-1}.}
$$

For our three-branch, two-step example, $L_2=\binom{4}{2}=6$, matching the
table. With two branches, the formula becomes $L_t=\binom{t+1}{1}=t+1$,
recovering the familiar binomial result. We compute one price and one
probability for each of these $L_t$ states.

> __Count states and distinct prices:__ The formula for $L_t$ counts branch-count
> vectors. Different vectors can sometimes give the same price. For example,
> if $f_1f_3=f_2^2$, the states $(1,0,1)$ and $(0,2,0)$ have the same price.
> We keep these states separate in the lattice and add their probabilities
> when computing the probability of that price.

Let's build an N-ary lattice from historical data and examine the distribution at
one of its levels.

> __Example:__
>
> [▶ Explore N-ary lattice models](CHEME-5660-L4a-Example-N-Ary-Lattice-Fall-2026.ipynb).
> Estimate several branch factors and probabilities from historical growth
> rates, build the recombining lattice, and compute its node prices and
> multinomial probabilities. We then check that the probabilities sum to one
> and plot the resulting distributions.

Using more branches lets us represent a one-step distribution with more
possible outcomes. With appropriate scaling of the price changes as the time step
shrinks, lattice models can approach continuous-time stochastic models of price.
We consider these models in the next lecture.

___

## Optional Advanced Material
The notebooks below extend today's material. They are optional and are not prerequisites for L4b; the [advanced index](advanced/README.md) lists them with a suggested order.

* [▶ First-passage exit rules](advanced/first-passage/CHEME-5660-L4a-Advanced-FirstPassage-ExitRules-Fall-2026.ipynb). What changes if we check the price after every lattice step and sell when a take-profit or stop-loss boundary is reached? We compute the probability of reaching either boundary first, or reaching neither during the holding period. We track open positions and compare with a calculation that checks only the final price. This explains why paths ending at the same price can produce different outcomes.
* [▶ Execution-aware probability of profit](advanced/execution/CHEME-5660-L4a-Advanced-ExecutionAware-ProbabilityOfProfit-Fall-2026.ipynb). How do trading costs change the probability of exceeding our target return? We extend the NPV calculation to include buying at the ask, selling at the bid, transaction fees, and an allowance for unfavorable execution prices (slippage). We derive the terminal price needed to exceed the target and compare the probability with the frictionless model. We examine how the individual costs, benchmark growth rate, and holding period affect the calculation.

___

## Summary
In this lecture, we developed an NPV-based framework for evaluating long stock positions and used the binomial lattice to compute the probability of exceeding a target return at a scheduled sale time. We then extended the model to allow multiple possible price movements at each step.

> __Key Takeaways__
>
> * **NPV-based trading framework**: We derived an expression for the net present value of a long stock position and scaled it by the initial investment to obtain a dimensionless discounted fractional return. This allowed us to account for the holding period and compare the present value of the sale proceeds with the purchase cost.
>
> * **Cumulative probability calculations for terminal targets**: We used the binomial lattice to compute the probability of exceeding a target scaled NPV at a scheduled sale time. We derived the minimum number of up moves needed to exceed the target and added the probabilities of the corresponding terminal nodes. The complementary probability describes finishing at or below the target.
>
> * **Extension to N-ary lattice models**: We generalized the binomial framework to allow $m$ possible price movements at each step. We used branch counts to calculate node prices and multinomial probabilities, and derived an expression for the number of states at each level. This gives us more flexibility in representing price movements, at the expense of additional computational cost.

In the next lecture, we will move from lattice models to continuous-time stochastic models of share-price dynamics.

___

## Disclaimer and Risks

__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products, or any investment or trading advice or strategy, is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.
